In [45]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rounakbanik/the-movies-dataset/ratings.csv
/kaggle/input/datasets/rounakbanik/the-movies-dataset/links_small.csv
/kaggle/input/datasets/rounakbanik/the-movies-dataset/credits.csv
/kaggle/input/datasets/rounakbanik/the-movies-dataset/keywords.csv
/kaggle/input/datasets/rounakbanik/the-movies-dataset/movies_metadata.csv
/kaggle/input/datasets/rounakbanik/the-movies-dataset/ratings_small.csv
/kaggle/input/datasets/rounakbanik/the-movies-dataset/links.csv


In [46]:
!pip install kagglehub scikit-surprise --quiet

import kagglehub
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print("Path to dataset files:", path)

import os
print(os.listdir(path))

Path to dataset files: /kaggle/input/datasets/rounakbanik/the-movies-dataset
['ratings.csv', 'links_small.csv', 'credits.csv', 'keywords.csv', 'movies_metadata.csv', 'ratings_small.csv', 'links.csv']


In [47]:
import pandas as pd
import numpy as np
from ast import literal_eval
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.snowball import SnowballStemmer
from surprise import Reader, Dataset, SVD, accuracy
from surprise.model_selection import cross_validate, train_test_split
from collections import defaultdict
import warnings; warnings.simplefilter('ignore')

import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [48]:
credits = pd.read_csv(f"{path}/credits.csv")
keywords = pd.read_csv(f"{path}/keywords.csv")
links_small = pd.read_csv(f"{path}/links_small.csv")
md = pd.read_csv(f"{path}/movies_metadata.csv", low_memory=False)
ratings = pd.read_csv(f"{path}/ratings_small.csv")

print("Movies metadata:", md.shape)
print("Ratings:", ratings.shape)
print("Credits:", credits.shape)
print("Keywords:", keywords.shape)

Movies metadata: (45466, 24)
Ratings: (100004, 4)
Credits: (45476, 3)
Keywords: (46419, 2)


## Stage 1: Data Cleaning & Merging

In [49]:
md['genres'] = md['genres'].fillna('[]').apply(literal_eval).apply(
    lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

def convert_int(x):
    try: return int(x)
    except: return np.nan

links_small = links_small[links_small['tmdbId'].notnull()]['tmdbId'].astype('int')

md['id'] = md['id'].apply(convert_int)
md = md[md['id'].notnull()]
md['id'] = md['id'].astype('int')

keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')

md = md.merge(credits, on='id')
md = md.merge(keywords, on='id')

smd = md[md['id'].isin(links_small)].copy()
print("Working subset shape:", smd.shape)

Working subset shape: (9219, 27)


## Stage 2: Content-Based Filtering (TF-IDF-style metadata + Cosine Similarity)

We combine genres, keywords, cast, and director into a single "soup" per movie,
then vectorize and compute pairwise similarity.

In [50]:
smd['cast'] = smd['cast'].apply(literal_eval)
smd['crew'] = smd['crew'].apply(literal_eval)
smd['keywords'] = smd['keywords'].apply(literal_eval)

def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

smd['director'] = smd['crew'].apply(get_director)
smd['cast'] = smd['cast'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])
smd['cast'] = smd['cast'].apply(lambda x: x[:3] if len(x) >= 3 else x)
smd['cast'] = smd['cast'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

smd['director'] = smd['director'].astype('str').apply(lambda x: str.lower(x.replace(" ", "")))
smd['director'] = smd['director'].apply(lambda x: [x, x, x])  # weighted 3x

smd['keywords'] = smd['keywords'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])
s = smd.apply(lambda x: pd.Series(x['keywords']), axis=1).stack().reset_index(level=1, drop=True)
s = s.value_counts()
s = s[s > 1]

stemmer = SnowballStemmer('english')
def filter_keywords(x):
    return [i for i in x if i in s]

smd['keywords'] = smd['keywords'].apply(filter_keywords)
smd['keywords'] = smd['keywords'].apply(lambda x: [stemmer.stem(i) for i in x])
smd['keywords'] = smd['keywords'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

smd['soup'] = smd['keywords'] + smd['cast'] + smd['director'] + smd['genres']
smd['soup'] = smd['soup'].apply(lambda x: ' '.join(x))

smd.head(2)[['title', 'soup']]

,title,soup
0,Toy Story,jealousi toy boy friendship friend rivalri boy...
1,Jumanji,boardgam disappear basedonchildren'sbook newho...


In [51]:
count = CountVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, stop_words='english')
count_matrix = count.fit_transform(smd['soup'])
cosine_sim = cosine_similarity(count_matrix, count_matrix)

smd = smd.reset_index(drop=True)
titles = smd['title']
indices = pd.Series(smd.index, index=smd['title'])

print("Similarity matrix shape:", cosine_sim.shape)

Similarity matrix shape: (9219, 9219)


In [52]:
# Build a case-insensitive lookup once
title_lookup = {t.lower(): t for t in titles}

def find_title(user_input):
    return title_lookup.get(user_input.strip().lower())

def get_recommendations(title, top_n=10):
    if title not in indices:
        print(f"'{title}' not found in dataset. Try another title.")
        return None
    
    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    return titles.iloc[movie_indices]

# --- User input ---
raw_input_title = input("Enter a movie title: ").strip()
matched_title = find_title(raw_input_title)

if matched_title is None:
    print(f"'{raw_input_title}' not found in dataset. Try checking spelling or capitalization.")
else:
    recommendations = get_recommendations(matched_title)
    if recommendations is not None:
        print(f"\nBecause you liked '{matched_title}', you might also like:\n")
        print(recommendations.to_string(index=False))

Enter a movie title:  joker



Because you liked 'Joker', you might also like:

                             Fados
               Different for Girls
                       Funny Felix
                           Drained
                   Fashion Victims
       The Last Days of Emma Blank
                 Ciao, Professore!
                      Hear My Song
                     The Big Tease
Vampire Girl vs. Frankenstein Girl


### Limitation
This content-based recommender returns the same result for every user who
queries the same movie — it has no notion of individual taste. Two people who
both liked "The Dark Knight" get identical recommendations, even if their
overall preferences differ. This motivates adding collaborative filtering.

## Stage 3: Collaborative Filtering (SVD)

SVD learns latent user and item factors from the ratings matrix to predict
how a specific user would rate a movie they haven't seen.

In [53]:
reader = Reader()
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

svd = SVD()
cv_results = cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

trainset = data.build_full_trainset()
svd.fit(trainset)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8937  0.8997  0.8974  0.9029  0.8821  0.8952  0.0072  
MAE (testset)     0.6864  0.6926  0.6890  0.6953  0.6814  0.6890  0.0048  
Fit time          1.09    1.11    1.13    1.15    1.15    1.13    0.02    
Test time         0.11    0.11    0.12    0.11    0.11    0.11    0.00    


In [54]:
svd.predict(1, 302)

Prediction(uid=1, iid=302, r_ui=None, est=np.float64(2.6473606252364728), details={'was_impossible': False})

## Stage 4: Hybrid Recommender

Combines content-based similarity (to shortlist relevant movies) with
SVD's personalized rating prediction (to rank them per user).

In [55]:
id_map = pd.read_csv(f"{path}/links_small.csv")[['movieId', 'tmdbId']]
id_map['tmdbId'] = id_map['tmdbId'].apply(convert_int)
id_map.columns = ['movieId', 'id']
id_map = id_map.merge(smd[['title', 'id']], on='id').set_index('title')
indices_map = id_map.set_index('id')

def hybrid(userId, title, top_n=10):
    if title not in indices:
        print(f"'{title}' not found in dataset. Try another title.")
        return None
    
    idx = indices[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
    
    sim_scores = list(enumerate(cosine_sim[int(idx)]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:26]
    movie_indices = [i[0] for i in sim_scores]

    movies = smd.iloc[movie_indices][['title', 'vote_count', 'vote_average', 'release_date', 'id']].copy()
    movies['est'] = movies['id'].apply(
        lambda x: svd.predict(userId, indices_map.loc[x]['movieId']).est
        if x in indices_map.index else np.nan
    )
    movies = movies.dropna(subset=['est']).sort_values('est', ascending=False)
    return movies.head(top_n)

# --- User input ---
raw_input_title = input("Enter a movie title: ").strip()
matched_title = find_title(raw_input_title)
user_id = input("Enter a user ID (try a number between 1-671): ").strip()

if matched_title is None:
    print(f"'{raw_input_title}' not found in dataset. Try checking spelling or capitalization.")
elif not user_id.isdigit():
    print("Please enter a valid numeric user ID.")
else:
    result = hybrid(int(user_id), matched_title)
    if result is not None:
        print(f"\nPersonalized recommendations for User {user_id} based on '{matched_title}':\n")
        display(result)

Enter a movie title:  gangster
Enter a user ID (try a number between 1-671):  61


'gangster' not found in dataset. Try checking spelling or capitalization.


In [56]:
print("User 1's personalized recommendations for 'Avatar':")
display(hybrid(1, 'Avatar'))

print("\nUser 5000's personalized recommendations for 'Avatar':")
display(hybrid(5000, 'Avatar'))

User 1's personalized recommendations for 'Avatar':


,title,vote_count,vote_average,release_date,id,est
999,The Terminator,4208.0,7.4,1984-10-26,218,3.164456
522,Terminator 2: Judgment Day,4274.0,7.7,1991-07-01,280,3.096609
962,Aliens,3282.0,7.7,1986-07-18,679,3.040815
910,The Abyss,822.0,7.1,1989-08-09,2756,2.978546
8357,Star Trek Into Darkness,4479.0,7.4,2013-05-05,54138,2.934652
2006,Fantastic Planet,140.0,7.6,1973-05-01,16306,2.907435
1660,Return from Witch Mountain,38.0,5.6,1978-03-10,14822,2.888448
344,True Lies,1138.0,6.8,1994-07-14,36955,2.838707
1613,Darby O'Gill and the Little People,35.0,6.7,1959-06-29,18887,2.814429
8622,X-Men: Days of Future Past,6155.0,7.5,2014-05-15,127585,2.801710



User 5000's personalized recommendations for 'Avatar':


,title,vote_count,vote_average,release_date,id,est
999,The Terminator,4208.0,7.4,1984-10-26,218,3.954404
962,Aliens,3282.0,7.7,1986-07-18,679,3.941356
522,Terminator 2: Judgment Day,4274.0,7.7,1991-07-01,280,3.938409
8622,X-Men: Days of Future Past,6155.0,7.5,2014-05-15,127585,3.807343
8357,Star Trek Into Darkness,4479.0,7.4,2013-05-05,54138,3.744174
910,The Abyss,822.0,7.1,1989-08-09,2756,3.716383
2006,Fantastic Planet,140.0,7.6,1973-05-01,16306,3.688621
1613,Darby O'Gill and the Little People,35.0,6.7,1959-06-29,18887,3.673424
1660,Return from Witch Mountain,38.0,5.6,1978-03-10,14822,3.603094
344,True Lies,1138.0,6.8,1994-07-14,36955,3.525307


## Stage 5: Evaluation

RMSE/MAE tell us how accurate individual rating predictions are.
Precision@K tells us how good the top-K ranked recommendations actually are —
a metric closer to what users experience.

In [57]:
def precision_recall_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = {}
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rec_k = sum(est >= threshold for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(
            (true_r >= threshold) and (est >= threshold)
            for (est, true_r) in user_ratings[:k]
        )
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
    return sum(precisions.values()) / len(precisions)

trainset2, testset2 = train_test_split(data, test_size=0.2, random_state=42)
svd_eval = SVD()
svd_eval.fit(trainset2)
predictions = svd_eval.test(testset2)

print("RMSE:", accuracy.rmse(predictions))
print("Precision@10:", precision_recall_at_k(predictions, k=10))

RMSE: 0.9010
RMSE: 0.9010171471746459
Precision@10: 0.7211157239845766


## Conclusion

- **Content-based filtering** captures similarity in genre/cast/director but
  gives identical results to every user.
- **Collaborative filtering (SVD)** personalizes predictions per user based on
  rating patterns, achieving an RMSE of ~0.89-0.90 on 5-fold cross-validation.
- **The hybrid model** combines both: content similarity shortlists relevant
  candidates, SVD ranks them by predicted rating for the specific user —
  giving personalized, content-relevant recommendations.

**Limitations:** cold-start problem for new users/movies with no ratings;
static dataset (2017 snapshot); precision@10 could be improved with feature
weighting or a larger ratings dataset.